# Check classifier and extract geomerty data of clusters
## Runtime: 14s

In [1]:
import numpy as np

from scipy.linalg import eigh
from scipy.stats import zscore
from scipy.interpolate import interp1d
from scipy.signal import find_peaks

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from matplotlib.patches import Circle
from matplotlib.patches import FancyArrowPatch

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

import pickle

import h5py

import os

In [2]:
def Angle_Calculation(vec1, vec2):
    # Compute dot product between vectors
    dot_product = np.dot(vec1, vec2)

    # Compute product of vector norms (magnitudes)
    norm_product = np.linalg.norm(vec1) * np.linalg.norm(vec2)

    # Avoid division by zero in case of a zero vector
    if norm_product == 0:
        return 0.0  # Or return an alternative fallback (e.g., 90.0) based on requirements

    # Compute cosine value and clip to [-1, 1] to handle floating-point inaccuracies
    cos_theta = dot_product / norm_product
    cos_theta = np.clip(
        cos_theta, -1.0, 1.0
    )  # Core fix: clamp values outside the valid domain

    # Compute angle in radians and convert to degrees
    angle_rad = np.arccos(cos_theta)
    angle_deg = angle_rad * (180 / np.pi)

    # Fold angle to the range [0, 90] degrees (for unoriented axial directions)
    min_angle = np.min([angle_deg, 180 - angle_deg])

    return min_angle

In [3]:
# 1. Define configuration for four cosmological simulations
simulations = [
    {
        "name": "SIMBA",
        "snapnum": "151",
        "save_dir": "/data/inspur_disk06/userdir/zhangzhuoming/Quench_Direction_Data/SIMBAm100n1024/data/",
    },
    {
        "name": "TNG100",
        "snapnum": "099",
        "save_dir": "/data/inspur_disk06/userdir/zhangzhuoming/Quench_Direction_Data/TNG100/data/",
    },
    {
        "name": "EAGLE",
        "snapnum": "028_z000p000",
        "save_dir": "/data/inspur_disk06/userdir/zhangzhuoming/Quench_Direction_Data/EAGLE/data/",
    },
    {
        "name": "SIMBAnofb",
        "snapnum": "151",
        "save_dir": "/data/inspur_disk06/userdir/zhangzhuoming/Quench_Direction_Data/SIMBAm50n512/nofb/data/",
    },
]

# Global parameters
M200c_min = 1e12
N_re = 50

# 2. Loop through each simulation for processing
for sim in simulations:
    sim_name = sim["name"]
    snapnum = sim["snapnum"]
    base_dir = sim["save_dir"]

    print(
        f"\n==================== Processing {sim_name} (snapnum: {snapnum}) ===================="
    )

    # Construct file paths
    star_pickle_path = os.path.join(
        base_dir, f"GalInHalo_star_axis/{snapnum}/CID_vec.pkl"
    )
    dm_pickle_path = os.path.join(
        base_dir, f"GalInHalo_dm_axis/{snapnum}/CID_vec.pkl"
    )
    galaxy_ave_path = os.path.join(
        base_dir, f"GalInHalo_star_axis/{snapnum}/galaxy_ave.npy"
    )

    # --- Step 1: Load pickle data ---
    with open(star_pickle_path, "rb") as f:
        CID_vec_var_star = pickle.load(f)

    with open(dm_pickle_path, "rb") as f:
        CID_vec_var_dm = pickle.load(f)

    #--- Step 2: Identify clusters with measured triaxial axes for stellar and dark matter components. ---
    star_CID = {item[0] for item in CID_vec_var_star}
    dm_CID = {item[0] for item in CID_vec_var_dm}

    # Select Cluster IDs (CID) with Successfully Measured Triaxial Axes for Both Stellar and Dark Matter Components
    common_CIDs = star_CID & dm_CID
    mismatched_count = len(star_CID ^ dm_CID)
    print(f"Number of mismatched halos: {mismatched_count}")

    CID_vec_var_star = [
        item for item in CID_vec_var_star if item[0] in common_CIDs
    ]
    CID_vec_var_dm = [item for item in CID_vec_var_dm if item[0] in common_CIDs]

    # Sort lists by CID to guarantee identical ordering
    CID_vec_var_star.sort(key=lambda x: x[0])
    CID_vec_var_dm.sort(key=lambda x: x[0])

    if len(CID_vec_var_star) != len(CID_vec_var_dm):
        raise ValueError(
            f"[{sim_name}] The number of clusters is inconsistent between star and dm!"
        )

    print("CID matching verified successfully.")

    # --- Step 3: Select central galaxy and filter by M200c mass ---
    Full_snap = np.load(galaxy_ave_path)
    mask_cgal = Full_snap[:, 8] == Full_snap[:, 9]
    cgal_data = Full_snap[mask_cgal, :]

    mask_cgal_M200c = cgal_data[:, 32] > M200c_min
    cgal_M_data = cgal_data[mask_cgal_M200c, :]
    cgal_M_CID = set(cgal_M_data[:, 8])

    # Extract the CID, Triaxial Axes Direction (star/dm), Variance (star/dm) of Selected clusters
    CID_vec_var_star = [
        sublist for sublist in CID_vec_var_star if sublist[0] in cgal_M_CID
    ]
    CID_vec_var_dm = [
        sublist for sublist in CID_vec_var_dm if sublist[0] in cgal_M_CID
    ]

    print(
        f"Number of clusters after M200c filtering: {len(CID_vec_var_star)}"
    )

    # --- Step 4: Calculate alignment angles and statistics ---
    all_data = []

    for i in range(len(CID_vec_var_star)):
        angle_i = [[], [], []]

        # Process the k-th (k=0, 1, 2) principal axis in the cluster
        for k in range(3):
            for m in range(N_re):
                axis_k_star = CID_vec_var_star[i][1][m][k]
                axis_k_dm = CID_vec_var_dm[i][1][m][k]
                angle_k = Angle_Calculation(axis_k_star, axis_k_dm)
                angle_i[k].append(angle_k)

        ave_angle_i = np.mean(angle_i, axis=1)
        std_angle_i = np.std(angle_i, axis=1)
        ave_vec_star = np.mean(CID_vec_var_star[i][1], axis=0)
        ave_var_star = np.mean(CID_vec_var_star[i][2], axis=0)
        ave_vec_dm = np.mean(CID_vec_var_dm[i][1], axis=0)
        ave_var_dm = np.mean(CID_vec_var_dm[i][2], axis=0)

        all_data.append(
            [
                CID_vec_var_star[i][0],
                ave_angle_i,
                std_angle_i,
                ave_vec_star,
                ave_vec_dm,
                ave_var_star,
                ave_var_dm,
            ]
        )

    # --- Step 5: Save processed output ---
    output_dir = f"cluster_data/{sim_name}/{snapnum}"
    os.makedirs(output_dir, exist_ok=True)

    output_file = os.path.join(output_dir, "all_data.pkl")
    with open(output_file, "wb") as f:
        pickle.dump(all_data, f)

    print(f"Output saved successfully to: {output_file}")


==================== Processing SIMBA (snapnum: 151) ====================
Number of mismatched halos: 0
CID matching verified successfully.
Number of clusters after M200c filtering: 3728
Output saved successfully to: cluster_data/SIMBA/151/all_data.pkl

==================== Processing TNG100 (snapnum: 099) ====================
Number of mismatched halos: 0
CID matching verified successfully.
Number of clusters after M200c filtering: 1716
Output saved successfully to: cluster_data/TNG100/099/all_data.pkl

==================== Processing EAGLE (snapnum: 028_z000p000) ====================
Number of mismatched halos: 0
CID matching verified successfully.
Number of clusters after M200c filtering: 984
Output saved successfully to: cluster_data/EAGLE/028_z000p000/all_data.pkl

==================== Processing SIMBAnofb (snapnum: 151) ====================
Number of mismatched halos: 0
CID matching verified successfully.
Number of clusters after M200c filtering: 548
Output saved successfully to